In [ ]:
# Comp kernel: Qwen3.6-35B-A3B BF16 on H100 for ARC-AGI-3 daily submission.
#
# This is the production submission kernel. Pre-reqs:
#   - cataluna84/qwen3-6-35b-a3b-bf16        (~71.93 GB safetensors)
#   - cataluna84/arc-agi-3-agents-pkg        (our agents/ package)
#   - cataluna84/arc-agi-3-transformers-wheels (transformers 5.7.0 + deps)
#   - competition data (arc-prize-2026-arc-agi-3)
#
# The kernel only does real eval inside the KAGGLE_IS_COMPETITION_RERUN env
# var; for save-runs / pre-submit checks it falls through to a dummy
# submission.parquet so the kernel still returns COMPLETE.
import os, sys, time, json, subprocess, shutil
from pathlib import Path

IS_RERUN = bool(os.environ.get('KAGGLE_IS_COMPETITION_RERUN'))
print('KAGGLE_IS_COMPETITION_RERUN =', IS_RERUN)

# --- 1. Locate the Qwen weights, agents pkg, transformers wheels -----------
_QWEN_CANDS = [
    Path('/kaggle/input/qwen3-6-35b-a3b-bf16'),
    Path('/kaggle/input/datasets/cataluna84/qwen3-6-35b-a3b-bf16'),
]
QWEN_DIR = next((c for c in _QWEN_CANDS if c.exists()), None)
if QWEN_DIR is None:
    for p in Path('/kaggle/input').rglob('config.json'):
        if p.parent.name.startswith('qwen3'):
            QWEN_DIR = p.parent
            break
print('Qwen weights at', QWEN_DIR)

_TX_CANDS = [
    Path('/kaggle/input/arc-agi-3-transformers-wheels'),
    Path('/kaggle/input/datasets/cataluna84/arc-agi-3-transformers-wheels'),
]
TX_WHEELS = next((c for c in _TX_CANDS if c.exists()), None)
if TX_WHEELS is None:
    for p in Path('/kaggle/input').rglob('transformers-*.whl'):
        TX_WHEELS = p.parent
        break
print('transformers wheels at', TX_WHEELS)

# --- 2. Install pillow (fix C-ext mismatch) + arc-agi from comp wheels -----
_WHEEL_CANDS = [
    Path('/kaggle/input/arc-prize-2026-arc-agi-3/arc_agi_3_wheels'),
    Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels'),
]
WHEELS = next((c for c in _WHEEL_CANDS if c.exists()), _WHEEL_CANDS[0])
print('competition wheels at', WHEELS)
if WHEELS.exists():
    PILLOW_TARGET = '/kaggle/working/_pillow_pkg'
    Path(PILLOW_TARGET).mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index',
                    f'--find-links={WHEELS}', '--upgrade',
                    '--target', PILLOW_TARGET, 'pillow'], check=False,
                   capture_output=True)
    if PILLOW_TARGET not in sys.path:
        sys.path.insert(0, PILLOW_TARGET)
    for mod in [m for m in list(sys.modules) if m.startswith('PIL')]:
        del sys.modules[mod]
    import PIL
    print(f'  PIL ok: {PIL.__version__} at {PIL.__file__}')
    # arc-agi + arcengine + python-dotenv from the comp wheels
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '--no-index', f'--find-links={WHEELS}',
                    'arc-agi', 'arcengine', 'python-dotenv'], check=False)
    print('  arc-agi + arcengine + python-dotenv installed')

# --- 3. Install transformers 5.7.0 from our offline-mirror Dataset ----------
if TX_WHEELS is not None:
    TX_TARGET = '/kaggle/working/_transformers_pkg'
    Path(TX_TARGET).mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index',
                    f'--find-links={TX_WHEELS}', '--upgrade',
                    '--target', TX_TARGET, '--no-deps',
                    'transformers', 'tokenizers', 'accelerate',
                    'huggingface_hub', 'safetensors',
                    'regex', 'filelock', 'fsspec', 'pyyaml', 'tqdm'],
                   check=False, capture_output=True)
    if TX_TARGET not in sys.path:
        sys.path.insert(0, TX_TARGET)
    for mod in [m for m in list(sys.modules) if m.split('.')[0] in {
        'transformers', 'tokenizers', 'accelerate', 'huggingface_hub',
        'safetensors'}]:
        del sys.modules[mod]
    import transformers as _tx
    print(f'  transformers ok: {_tx.__version__} at {_tx.__file__}')

# --- 4. Copy our agents/ package into /kaggle/working/qwen_agent_lib.py ----
# We don't put it under agents/ because the official ARC-AGI-3-Agents repo
# (copied below) has its OWN agents/ package and we don't want to clobber it.
_AGENT_CANDS = [
    Path('/kaggle/input/arc-agi-3-agents-pkg/agents'),
    Path('/kaggle/input/datasets/cataluna84/arc-agi-3-agents-pkg/agents'),
]
ARC_AGENTS_SRC = next((c for c in _AGENT_CANDS if c.exists()), None)
if ARC_AGENTS_SRC is None:
    for p in Path('/kaggle/input').rglob('qwen_agent.py'):
        ARC_AGENTS_SRC = p.parent
        break
print('our agents/ at', ARC_AGENTS_SRC)

QWEN_LIB = Path('/kaggle/working/qwen_agent_lib.py')
if ARC_AGENTS_SRC is not None and not QWEN_LIB.exists():
    src = (ARC_AGENTS_SRC / 'qwen_agent.py').read_text()
    # Replace 'from agents import ...' with explicit fallback that does NOT
    # depend on a top-level agents/ package (which on Kaggle is the upstream
    # ARC-AGI-3-Agents repo, not ours).
    src = src.replace(
        'from agents import GameAction, GameState',
        'from arcengine import GameAction, GameState',
    )
    QWEN_LIB.write_text(src)
    print(f'wrote {QWEN_LIB} ({QWEN_LIB.stat().st_size} bytes)')

# --- 5. Set agent env vars (model path, dtype, decode budget) --------------
os.environ['QWEN_MODEL_PATH']     = str(QWEN_DIR) if QWEN_DIR else ''
os.environ['QWEN_DTYPE']          = 'bf16'
os.environ['QWEN_DEVICE_MAP']     = 'auto'
os.environ['QWEN_MAX_NEW_TOKENS'] = '16'
os.environ['QWEN_HISTORY_LEN']    = '8'

print('--- setup done at', time.strftime('%H:%M:%S'), '---')


In [ ]:
# %%writefile-style emit of /kaggle/working/my_agent.py.
#
# This is the SHIM seen by the official ARC-AGI-3-Agents harness:
#   class MyAgent(Agent): super().__init__; choose_action(frames, lf) -> GameAction
#
# Internally it delegates to our QwenAgent (loaded once as a module-level
# singleton across all games to amortize the ~580s model load).
MY_AGENT_PY = '''
from __future__ import annotations

import os
import sys
import time
from typing import Any

# Make our overlay paths importable BEFORE any heavy imports.
for _p in ("/kaggle/working/_pillow_pkg",
          "/kaggle/working/_transformers_pkg",
          "/kaggle/working"):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from agents.agent import Agent  # official base class from ARC-AGI-3-Agents
from arcengine import FrameData, GameAction, GameState

# Lazy import of our QwenAgent (defers the heavy torch/transformers chain)
_QWEN_SINGLETON: Any = None

def _get_qwen(arc_env: Any, game_id: str) -> Any:
    global _QWEN_SINGLETON
    if _QWEN_SINGLETON is None:
        from qwen_agent_lib import QwenAgent  # noqa: WPS433
        _QWEN_SINGLETON = QwenAgent(arc_env=arc_env, game_id=game_id)
        # Force-load now so the cost is incurred outside the per-action loop
        t0 = time.time()
        _QWEN_SINGLETON._ensure_model_loaded()
        print(f"[my_agent] QwenAgent loaded in {time.time()-t0:.1f}s")
    else:
        # Per-game state reset (history, frame-change cache)
        _QWEN_SINGLETON.game_id = game_id
        _QWEN_SINGLETON._history.clear()
        _QWEN_SINGLETON._prev_grid_hash = 0
        _QWEN_SINGLETON._prev_action_name = None
    return _QWEN_SINGLETON


class MyAgent(Agent):
    \"\"\"Official-API wrapper around our QwenAgent.

    The Kaggle eval harness instantiates MyAgent once per game; we share the
    35B-A3B weights across all games via the module-level _QWEN_SINGLETON to
    avoid paying the ~580s safetensors load 25 times.
    \"\"\"

    MAX_ACTIONS = float("inf")

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self._impl = _get_qwen(arc_env=self.arc_env, game_id=self.game_id)

    def choose_action(self, frames: list, lf: FrameData) -> GameAction:
        try:
            action = self._impl.choose_action(lf)
            action.reasoning = {"agent": "qwen", "hist": list(self._impl._history)[-3:]}
            return action
        except Exception as e:
            import traceback
            traceback.print_exc()
            # Defensive fallback so the eval doesn't get stuck on errors.
            avail = [int(a) for a in (getattr(lf, "available_actions", None) or [])]
            fallback_id = sorted(avail)[0] if avail else 1
            a = GameAction.from_id(fallback_id)
            a.reasoning = {"agent": "qwen", "err": f"{type(e).__name__}: {e}"}
            return a
'''
Path('/kaggle/working/my_agent.py').write_text(MY_AGENT_PY)
print('wrote /kaggle/working/my_agent.py:', Path('/kaggle/working/my_agent.py').stat().st_size, 'bytes')


In [ ]:
# Run the official ARC-AGI-3-Agents harness (online mode, talks to gateway:8001)
# only when KAGGLE_IS_COMPETITION_RERUN is set. For save-test runs this cell
# is a no-op - we still emit a dummy submission.parquet in the next cell.
import os
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import subprocess
    print('KAGGLE_IS_COMPETITION_RERUN=1, running official ARC-AGI-3-Agents harness')
    # Prime the gateway (curl with retries; gateway can take a few sec to come up)
    subprocess.run(['curl', '--fail', '--retry', '999', '--retry-all-errors',
                    '--retry-delay', '5', '--retry-max-time', '600',
                    'http://gateway:8001/api/games'], check=False)
    # Copy the harness from the comp data into /kaggle/working
    if not os.path.isdir('/kaggle/working/ARC-AGI-3-Agents'):
        subprocess.run(['cp', '-r',
                        '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents',
                        '/kaggle/working/ARC-AGI-3-Agents'], check=False)
    # Drop our agent into the templates dir
    subprocess.run(['cp', '/kaggle/working/my_agent.py',
                    '/kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py'], check=False)
    # Also copy our qwen_agent_lib.py so the harness can find it on its sys.path
    subprocess.run(['cp', '/kaggle/working/qwen_agent_lib.py',
                    '/kaggle/working/ARC-AGI-3-Agents/qwen_agent_lib.py'], check=False)
    # Register MyAgent in the harness's agents/__init__.py
    init_py = '''from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent
load_dotenv()
AVAILABLE_AGENTS: dict[str, Type[Agent]] = {"random": Random, "myagent": MyAgent}
'''
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as fh:
        fh.write(init_py)
    # Write the .env (gateway-online mode for the harness)
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as fh:
        fh.write('SCHEME=http\n'
                 'HOST=gateway\n'
                 'PORT=8001\n'
                 'ARC_API_KEY=test-key-123\n'
                 'ARC_BASE_URL=http://gateway:8001/\n'
                 'OPERATION_MODE=online\n'
                 'RECORDINGS_DIR=/kaggle/working/server_recording\n')
    print('starting main.py --agent myagent')
    rc = subprocess.run(['python', 'main.py', '--agent', 'myagent'],
                        cwd='/kaggle/working/ARC-AGI-3-Agents',
                        env={**os.environ, 'MPLBACKEND': 'agg',
                             'PYTHONPATH': '/kaggle/working:/kaggle/working/_pillow_pkg:/kaggle/working/_transformers_pkg'})
    print(f'main.py rc={rc.returncode}')
else:
    print('not in competition rerun; skipping harness')


In [ ]:
# Always write a dummy submission.parquet in non-rerun (save-test / local-eval)
# context so the kernel completes successfully and the submission pipeline
# stays unblocked. The Kaggle eval will overwrite this with the real one
# during the rerun.
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'],
    )
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    print('wrote dummy /kaggle/working/submission.parquet')
